# Đánh giá PaddleOCR trên golden set tổng hợp

Chạy PaddleOCR lên các giấy tờ do `src/data-golden-set-generator` sinh ra, so với `ground_truth.json` để xem:

1. **End-to-end** (detection + recognition trên cả trang): mỗi trường có đọc ra đúng không, theo mức nhiễu `clean → light → medium → heavy`.
2. **Recognition-only** (cắt đúng bbox ground truth rồi nhận dạng): tách lỗi do model nhận dạng với lỗi do detection.
3. **Ký tự hay sai**: tiếng Việt có dấu chồng (ọ, ữ, ệ…) là điểm yếu cần xem kỹ.

**Chỉ số:**

- **CER** = khoảng cách Levenshtein / số ký tự ground truth. End-to-end dùng khớp bán toàn cục: lấy chuỗi con khớp nhất trong các dòng OCR chồng lên bbox trường, vì dòng OCR thường dính cả nhãn (`Họ và tên: ...`).
- **CER bỏ dấu**: CER sau khi bỏ dấu và chữ hoa. Chênh lệch giữa hai CER là phần lỗi do dấu.
- **Khớp đúng**: tỷ lệ trường có CER = 0.
- **Detection recall**: tỷ lệ trường có ít nhất một dòng OCR chồng lên bbox.

Trường `checkbox` bị loại, OCR không đọc ô tích.

> Đây là dữ liệu tổng hợp, metric sẽ đẹp hơn hồ sơ thật. Dùng để so model và mức nhiễu, không dùng làm con số cuối.

## 0. Cài đặt (chỉ cần trên Colab)

Máy local dùng `jupyters/.venv` (xem `README.md`). Trên Colab, chọn runtime GPU rồi chạy cell dưới.

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q paddlepaddle-gpu==3.3.1 -i https://www.paddlepaddle.org.cn/packages/stable/cu126/
    %pip install -q paddleocr==3.7.0 rapidfuzz

## 1. Cấu hình

Dữ liệu để trong thư mục `data/` cạnh notebook, cấu trúc bên trong tùy ý. Notebook quét đệ quy mọi ảnh trong `data/` để đưa vào model.

- **Local:** `ln -s ../src/data-golden-set-generator/generated data`
- **Colab:** upload file zip rồi giải nén vào `/content/data` (hoặc đổi `DATA_DIR` sang thư mục trên Google Drive).

Ảnh nào có trong một `ground_truth.json` (do generator sinh) thì được chấm điểm, ảnh khác chỉ chạy OCR.

In [ ]:
import os
os.environ.setdefault("PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK", "True")

import json, time, unicodedata
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rapidfuzz.distance import Levenshtein

DATA_DIR = Path("data")                  # tương đối với thư mục notebook (trên Colab là /content)
CACHE_DIR = Path("outputs/paddleocr")    # kết quả OCR, chạy lại không tốn thời gian
IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp", ".webp"}
SKIP_SUFFIXES = ("_bbox",)               # ảnh debug vẽ bbox của generator

MAX_IMAGES = 24                          # None = tất cả. CPU ~25 giây/ảnh/model, GPU nhanh hơn nhiều
LEVELS = ["clean", "light", "medium", "heavy"]
DET_MODEL = "PP-OCRv6_medium_det"        # giữ cố định để so riêng model nhận dạng
REC_MODELS = [
    "PP-OCRv6_medium_rec",               # model PaddleOCR chọn cho lang="vi"
    "latin_PP-OCRv5_mobile_rec",
    "PP-OCRv5_server_rec",
]
import paddle
if paddle.device.is_compiled_with_cuda() and paddle.device.cuda.device_count() > 0:
    PADDLE_KWARGS = dict(device="gpu:0")
else:
    # Paddle CPU bản 3.x lỗi oneDNN (ConvertPirAttribute2RuntimeAttribute) với model v6, nên tắt
    PADDLE_KWARGS = dict(device="cpu", enable_mkldnn=False)
print(PADDLE_KWARGS)

pd.set_option("display.max_colwidth", 80)

## 2. Tìm ảnh và ground truth

Quét đệ quy `DATA_DIR` lấy mọi ảnh. Ground truth tra theo đường dẫn ảnh từ mọi `ground_truth.json` tìm được. Mỗi dòng của `items` là một trường trên một ảnh. Bbox của ảnh nhiễu đã được generator biến đổi theo phép xoay/phối cảnh.

In [ ]:
if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Không thấy {DATA_DIR.resolve()}. Xem mục 1 để đặt dữ liệu.")

images = sorted(
    p.as_posix() for p in DATA_DIR.rglob("*")
    if p.suffix.lower() in IMAGE_EXTS and not p.stem.endswith(SKIP_SUFFIXES)
)[:MAX_IMAGES]

def index_ground_truth(data_dir: Path) -> dict:
    """Đường dẫn ảnh -> (thông tin giấy tờ, level, page, bbox theo tên trường)."""
    index = {}
    for gt_path in data_dir.rglob("ground_truth.json"):
        gt = json.loads(gt_path.read_text(encoding="utf-8"))
        for doc in gt["documents"]:
            clean_boxes = {f["name"]: f.get("bbox", []) for f in doc["fields"]}
            pages = [("clean", i + 1, p, clean_boxes) for i, p in enumerate(doc["files"]["pages"])]
            pages += [(n["level"], n["page"], n["path"], n["bboxes"]) for n in doc["files"].get("noisy", [])]
            for level, page, rel, boxes in pages:
                index[(gt_path.parent / rel).as_posix()] = (gt["dossier_id"], doc, level, page, boxes)
    return index

def build_items(images, gt_index) -> pd.DataFrame:
    rows = []
    for im in images:
        if im not in gt_index:
            continue
        dossier, doc, level, page, boxes = gt_index[im]
        for f in doc["fields"]:
            if f.get("fill") == "checkbox" or not f["text"].strip():
                continue
            for k, b in enumerate(bb for bb in boxes.get(f["name"], []) if bb.get("page", 1) == page):
                rows.append(dict(
                    dossier=dossier, doc_type=doc["doc_type"], level=level, image=im,
                    field=f["name"], field_type=f["type"], fill=f.get("fill", "printed"),
                    occurrence=k, truth=f["text"], x=b["x"], y=b["y"], w=b["w"], h=b["h"],
                ))
    return pd.DataFrame(rows)

gt_index = index_ground_truth(DATA_DIR)
items = build_items(images, gt_index)
unlabeled = [im for im in images if im not in gt_index]
print(f"{len(images)} ảnh: {len(images) - len(unlabeled)} có ground truth ({len(items)} trường), {len(unlabeled)} chỉ chạy OCR")
if items.empty:
    raise RuntimeError("Không có ảnh nào có ground truth để chấm điểm")
items["level"] = pd.Categorical(items["level"], LEVELS, ordered=True)
items.groupby(["doc_type", "fill"]).field.unique().to_frame("fields")

Xem thử một ảnh nhiễu nặng cạnh bản sạch, kèm bbox ground truth:

In [ ]:
def draw_boxes(image_path, df, color=(214, 89, 38)):
    img = cv2.cvtColor(cv2.imread(image_path), cv2.COLOR_BGR2RGB)
    for r in df.itertuples():
        cv2.rectangle(img, (int(r.x), int(r.y)), (int(r.x + r.w), int(r.y + r.h)), color, 3)
    return img

sample = items[items.dossier == items.dossier.iloc[0]]
first_doc = sample.doc_type.iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(14, 10))
for ax, level in zip(axes, ["clean", "heavy"]):
    s = sample[(sample.doc_type == first_doc) & (sample.level == level)]
    if len(s):
        ax.imshow(draw_boxes(s.image.iloc[0], s))
    ax.set_title(f"{first_doc} - {level}")
    ax.axis("off")
plt.tight_layout()

## 3. Chạy OCR cả trang

Kết quả mỗi ảnh được cache thành JSON trong `outputs/paddleocr/<model>/`. Xóa thư mục cache để chạy lại.

In [ ]:
from paddleocr import PaddleOCR

def cache_path(rec_model, image):
    return CACHE_DIR / rec_model / Path(image).relative_to(DATA_DIR).with_suffix(".json")

def run_pipeline(rec_model, image_list):
    todo = [im for im in image_list if not cache_path(rec_model, im).exists()]
    print(f"{rec_model}: {len(image_list) - len(todo)} ảnh đã cache, chạy {len(todo)} ảnh")
    if not todo:
        return
    ocr = PaddleOCR(
        text_detection_model_name=DET_MODEL,
        text_recognition_model_name=rec_model,
        use_doc_orientation_classify=False,
        use_doc_unwarping=False,
        use_textline_orientation=False,
        **PADDLE_KWARGS,
    )
    for i, im in enumerate(todo, 1):
        t0 = time.perf_counter()
        r = ocr.predict(im)[0]
        out = dict(
            seconds=time.perf_counter() - t0,
            lines=[dict(text=t, score=float(s), box=[int(v) for v in b])
                   for t, s, b in zip(r["rec_texts"], r["rec_scores"], r["rec_boxes"])],
        )
        cp = cache_path(rec_model, im)
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(out, ensure_ascii=False), encoding="utf-8")
        if i % 10 == 0 or i == len(todo):
            print(f"  {i}/{len(todo)} ({out['seconds']:.1f}s/ảnh)")
    del ocr

for rec_model in REC_MODELS:
    run_pipeline(rec_model, images)

Ảnh không có ground truth: in text OCR để xem bằng mắt.

In [ ]:
for im in unlabeled[:5]:
    print(f"\n=== {im} ===")
    for m in REC_MODELS:
        lines = json.loads(cache_path(m, im).read_text(encoding="utf-8"))["lines"]
        print(f"--- {m}")
        print("\n".join(f"{l['score']:.2f}  {l['text']}" for l in lines))

## 4. Chấm điểm end-to-end

In [ ]:
def norm(s: str) -> str:
    return " ".join(unicodedata.normalize("NFC", s).split())

def strip_marks(s: str) -> str:
    s = unicodedata.normalize("NFD", s.replace("đ", "d").replace("Đ", "D"))
    return "".join(c for c in s if unicodedata.category(c) != "Mn").casefold()

def semiglobal_distance(gt: str, hyp: str) -> int:
    """Số phép sửa ít nhất để gt thành một chuỗi con nào đó của hyp."""
    if not gt:
        return 0
    prev = [0] * (len(hyp) + 1)
    for i, g in enumerate(gt, 1):
        cur = [i] + [0] * len(hyp)
        for j, h in enumerate(hyp, 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (g != h))
        prev = cur
    return min(prev)

def overlap_lines(lines, x, y, w, h, min_cover=0.5):
    """Dòng OCR mà phần giao chiếm >= min_cover diện tích của bên nhỏ hơn (trường hoặc dòng)."""
    hits = []
    for ln in lines:
        x0, y0, x1, y1 = ln["box"]
        iw = min(x + w, x1) - max(x, x0)
        ih = min(y + h, y1) - max(y, y0)
        if iw <= 0 or ih <= 0:
            continue
        if iw * ih / min(w * h, (x1 - x0) * (y1 - y0)) >= min_cover:
            hits.append(ln)
    return sorted(hits, key=lambda l: ((l["box"][1] + l["box"][3]) / 2, l["box"][0]))

def score_e2e(rec_model):
    ocr_cache = {im: json.loads(cache_path(rec_model, im).read_text(encoding="utf-8")) for im in items.image.unique()}
    out = []
    for r in items.itertuples():
        hits = overlap_lines(ocr_cache[r.image]["lines"], r.x, r.y, r.w, r.h)
        hyp = norm(" ".join(l["text"] for l in hits))
        gt = norm(r.truth)
        out.append(dict(
            detected=bool(hits), hyp=hyp,
            cer=semiglobal_distance(gt, hyp) / len(gt),
            cer_nomark=semiglobal_distance(strip_marks(gt), strip_marks(hyp)) / len(gt),
        ))
    return items.assign(model=rec_model, **pd.DataFrame(out, index=items.index))

e2e = pd.concat([score_e2e(m) for m in REC_MODELS], ignore_index=True)
e2e["exact"] = e2e.cer == 0
e2e["cer"] = e2e.cer.clip(upper=1)
e2e["cer_nomark"] = e2e.cer_nomark.clip(upper=1)

seconds = pd.DataFrame(
    [dict(model=m, seconds=json.loads(cache_path(m, im).read_text())["seconds"]) for m in REC_MODELS for im in images]
).groupby("model").seconds.mean().rename("giây/ảnh")

summary = (e2e.groupby("model")
           .agg(CER=("cer", "mean"), CER_bo_dau=("cer_nomark", "mean"), khop_dung=("exact", "mean"), det_recall=("detected", "mean"))
           .join(seconds).loc[REC_MODELS])
summary.style.format({"CER": "{:.1%}", "CER_bo_dau": "{:.1%}", "khop_dung": "{:.1%}", "det_recall": "{:.1%}", "giây/ảnh": "{:.1f}"})

### CER theo mức nhiễu, tách chữ viết tay và chữ in

In [ ]:
# Bảng màu categorical mặc định (dataviz skill), gán cố định theo model, không theo thứ hạng
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]
MODEL_COLOR = {m: PALETTE[i] for i, m in enumerate(REC_MODELS)}
INK, MUTED, GRID = "#1f2328", "#59636e", "#d8dee4"

def style_axes(ax):
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(GRID)
    ax.tick_params(colors=MUTED)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

def plot_by_level(df, metric, title):
    fills = [f for f in ["handwritten", "printed"] if f in set(df.fill)]
    agg = df.groupby(["fill", "model", "level"], observed=True)[metric].mean()
    fig, axes = plt.subplots(1, len(fills), figsize=(6 * len(fills), 4), sharey=True, squeeze=False)
    for ax, fill in zip(axes[0], fills):
        for m in REC_MODELS:
            s = agg.loc[(fill, m)].reindex(LEVELS)
            ax.plot(LEVELS, s.values, color=MODEL_COLOR[m], linewidth=2, marker="o", markersize=8,
                    markeredgecolor="white", markeredgewidth=2, label=m)
            ax.annotate(f"{s.values[-1]:.0%}", (len(LEVELS) - 1, s.values[-1]), xytext=(8, 0),
                        textcoords="offset points", va="center", color=INK, fontsize=9)
        ax.set_title(fill, color=INK, loc="left")
        ax.set_ylim(bottom=0)
        style_axes(ax)
    axes[0][0].set_ylabel(title, color=MUTED)
    axes[0][-1].legend(frameon=False, loc="upper left", bbox_to_anchor=(1.08, 1))
    fig.suptitle(title, x=0.01, ha="left", color=INK)
    plt.tight_layout()

plot_by_level(e2e, "cer", "CER end-to-end (thấp hơn là tốt hơn)")

In [ ]:
(e2e.pivot_table(index=["fill", "level"], columns="model", values=["cer", "cer_nomark"], aggfunc="mean", observed=True)
    .reindex(columns=REC_MODELS, level=1)
    .style.format("{:.1%}").background_gradient(cmap="Oranges", vmin=0, vmax=0.5))

### CER theo trường

Trường nào khó nhất (gộp mọi mức nhiễu):

In [ ]:
(e2e.pivot_table(index=["doc_type", "field", "fill"], columns="model", values="cer", aggfunc="mean", observed=True)
    [REC_MODELS].sort_values(REC_MODELS[0], ascending=False)
    .style.format("{:.1%}").background_gradient(cmap="Oranges", vmin=0, vmax=0.5))

## 5. Recognition-only trên vùng cắt theo bbox ground truth

Cắt đúng bbox trường (nới 6px) rồi cho model nhận dạng đọc trực tiếp. Nếu CER ở đây thấp mà end-to-end cao thì lỗi nằm ở detection hoặc ở cách ghép dòng.

In [ ]:
from paddleocr import TextRecognition

PAD = 6

def crop(img, r):
    H, W = img.shape[:2]
    x0, y0 = max(int(r.x) - PAD, 0), max(int(r.y) - PAD, 0)
    x1, y1 = min(int(r.x + r.w) + PAD, W), min(int(r.y + r.h) + PAD, H)
    return img[y0:y1, x0:x1]

def run_rec_only(rec_model):
    cp = CACHE_DIR / rec_model / "_rec_only.json"
    cache = json.loads(cp.read_text(encoding="utf-8")) if cp.exists() else {}
    keys = [f"{Path(r.image).relative_to(DATA_DIR).as_posix()}|{r.field}|{r.occurrence}" for r in items.itertuples()]
    todo = [(k, r) for k, r in zip(keys, items.itertuples()) if k not in cache]
    print(f"{rec_model}: chạy {len(todo)} vùng cắt")
    if todo:
        model = TextRecognition(model_name=rec_model, **PADDLE_KWARGS)
        img_cache = {}
        for k, r in todo:
            img = img_cache.setdefault(r.image, cv2.imread(r.image))
            res = model.predict(crop(img, r))[0]
            cache[k] = dict(text=res["rec_text"], score=float(res["rec_score"]))
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(cache, ensure_ascii=False), encoding="utf-8")
        del model
    hyp = [norm(cache[k]["text"]) for k in keys]
    gt = [norm(g) for g in items.truth]
    return items.assign(
        model=rec_model, hyp=hyp,
        cer=[min(Levenshtein.distance(g, h) / len(g), 1) for g, h in zip(gt, hyp)],
        cer_nomark=[min(Levenshtein.distance(strip_marks(g), strip_marks(h)) / len(g), 1) for g, h in zip(gt, hyp)],
    )

rec = pd.concat([run_rec_only(m) for m in REC_MODELS], ignore_index=True)
rec["exact"] = rec.cer == 0

both = pd.concat([
    e2e.groupby("model").agg(CER=("cer", "mean"), CER_bo_dau=("cer_nomark", "mean"), khop_dung=("exact", "mean")).assign(che_do="end-to-end"),
    rec.groupby("model").agg(CER=("cer", "mean"), CER_bo_dau=("cer_nomark", "mean"), khop_dung=("exact", "mean")).assign(che_do="rec-only"),
]).reset_index().set_index(["model", "che_do"]).loc[REC_MODELS]
both.style.format("{:.1%}")

In [ ]:
plot_by_level(rec, "cer", "CER recognition-only trên vùng cắt")

## 6. Ký tự hay bị đọc sai

Đếm trên kết quả recognition-only: mỗi ký tự ground truth bị xóa hoặc thay bằng ký tự khác. Tỷ lệ = số lần sai / số lần xuất hiện.

In [ ]:
from collections import Counter

def char_errors(df):
    seen, wrong, became = Counter(), Counter(), {}
    for g, h in zip(df.truth.map(norm), df.hyp):
        seen.update(g)
        for op in Levenshtein.editops(g, h):
            if op.tag in ("delete", "replace"):
                c = g[op.src_pos]
                wrong[c] += 1
                became.setdefault(c, Counter())[h[op.dest_pos] if op.tag == "replace" else "∅"] += 1
    rows = [dict(char=c, xuat_hien=seen[c], sai=wrong[c], ti_le=wrong[c] / seen[c],
                 thanh=", ".join(f"{k}×{v}" for k, v in became[c].most_common(3)))
            for c in wrong if c != " "]
    return pd.DataFrame(rows).sort_values(["ti_le", "sai"], ascending=False)

for m in REC_MODELS:
    print(f"\n=== {m} (chỉ ảnh sạch) ===")
    display(char_errors(rec[(rec.model == m) & (rec.level == "clean")]).head(15).style.format({"ti_le": "{:.0%}"}).hide(axis="index"))

## 7. Xem các ca sai nặng nhất

In [ ]:
def show_worst(df, model, n=12, level=None):
    s = df[(df.model == model) & (df.cer > 0)]
    if level:
        s = s[s.level == level]
    s = s.sort_values("cer", ascending=False).head(n)
    if s.empty:
        print("Không có ca sai")
        return
    fig, axes = plt.subplots(len(s), 1, figsize=(10, 1.1 * len(s)))
    axes = np.atleast_1d(axes)
    for ax, r in zip(axes, s.itertuples()):
        img = cv2.cvtColor(crop(cv2.imread(r.image), r), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"[{r.level}/{r.fill}] {r.field}  CER {r.cer:.0%}\nGT : {r.truth}\nOCR: {r.hyp}",
                     loc="left", fontsize=9, color=INK, family="monospace")
    plt.tight_layout()

show_worst(rec, REC_MODELS[0], level="clean")

In [ ]:
show_worst(rec, REC_MODELS[0], level="heavy")

## 8. Ghi chú đọc kết quả

- Model nào có CER bỏ dấu thấp nhưng CER cao: đọc được mặt chữ, hỏng ở dấu. Với tiếng Việt, cần model có từ điển đủ ký tự có dấu chồng.
- CER end-to-end cao hơn rec-only nhiều: kiểm tra detection (dòng bị cắt đôi, dòng dính vào nhau) và ngưỡng `min_cover` trong `overlap_lines`.
- Chữ viết tay ở golden set là font viết tay, dễ hơn chữ viết tay thật. Cần kiểm lại trên `real/` trước khi chốt model.